# Spatial verification run — clean `repro` package (branch `rebuttal`)
**Upload-and-run on a GPU runtime.** One implementation, no vendored `src`;
every knob in `repro/configs/spatial.yaml`; every trainer seeds BEFORE
model construction.

Runs the full published protocol: 3 scenes x 5 seeds x 12-theta sweep,
12 detectors (DART, DART-CFAR, DARTS, DARTS-CFAR, AMF-global, AMF-local,
GMM-Levin, LRao(val-ES) + THANTD, HTD-Net, TSTTD, OS-VAE). Saves raw
scores per (seed, theta), per-seed metrics (per-class Pfa on Pavia),
checkpoints (LRao every 10 epochs), then **verifies against the published
scores per (seed, theta), per-seed metrics (per-class Pfa on Pavia),
checkpoints (LRao every 10 epochs), then **verifies against the published
numbers** and zips everything.

**Deep baselines load the bundled published checkpoints**
(`repro/checkpoints/deep/`, 60 files) instead of retraining (hours).
Set `retrain_deep: true` in the config cell to retrain from scratch.

In [6]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, sys, torch
sys.path.insert(0, '.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
for p in ('repro/data/pavia-u.mat', 'repro/data/Sandiego.mat',
          'repro/data/Sandiego2.mat', 'repro/data/sandiego_regions.json',
          'repro/data/sandiego2_regions.json'):
    assert os.path.exists(p), f'missing {p}'
print('all bundled data present')

Cloning into 'repo'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 152 (delta 3), reused 131 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 157.89 MiB | 26.97 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/Users/shirschneorson/PycharmProjects/final-paper-experiment/repo/repo
device: cpu
all bundled data present


In [9]:
from repro.protocols import spatial as SP
CFG = SP.load_cfg()
print('scenes:', CFG['scenes'], ' seeds:', CFG['seeds'])
print('thetas:', CFG['thetas'])
print('deep:', CFG['deep_detectors'], '| pretrained:',
      CFG['deep_pretrained'], '| retrain_deep:', CFG['retrain_deep'])
# CFG['retrain_deep'] = True   # <- uncomment to retrain the deep baselines


scenes: ['pavia4', 'sandiego', 'sandiego2']  seeds: [42, 43, 44, 45, 46]
thetas: [0.03, 0.075, 0.15, 0.225, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
deep: ['THANTD', 'HTDNet', 'TSTTD', 'OSVAE'] | pretrained: repro/checkpoints/deep | retrain_deep: False


In [ ]:
SP.run_scene('pavia4', CFG, out_root='results/spatial', device=DEVICE)

[pavia4] train 4026 px (66, 61)  test 7476 px (89, 84)  ||s||=14357.9
[pavia4] seed 42


    [LRao] best val epoch 1000 (val -0.3781, stopped 1000, fit 3221/val 805)
    resumed repro/checkpoints/deep/THANTD_pavia4_seed42.pt
    resumed repro/checkpoints/deep/HTDNet_pavia4_seed42.pt
    resumed repro/checkpoints/deep/TSTTD_pavia4_seed42.pt
    resumed repro/checkpoints/deep/OSVAE_pavia4_seed42.pt


In [ ]:
SP.run_scene('sandiego', CFG, out_root='results/spatial', device=DEVICE)

[sandiego] train (4020, 189) test (1860, 189) ||s||=39598.9  (archive-exact)
[sandiego] train 4020 px (60, 67)  test 1860 px (60, 31)  ||s||=39598.9
[sandiego] seed 42


    [LRao] best val epoch 1000 (val -0.8709, stopped 1000, fit 3216/val 804)
    resumed repro/checkpoints/deep/THANTD_sandiego_seed42.pt
    resumed repro/checkpoints/deep/HTDNet_sandiego_seed42.pt
    resumed repro/checkpoints/deep/TSTTD_sandiego_seed42.pt
    resumed repro/checkpoints/deep/OSVAE_sandiego_seed42.pt


In [ ]:
SP.run_scene('sandiego2', CFG, out_root='results/spatial', device=DEVICE)

[sandiego2] train (4455, 189) test (2226, 189) ||s||=27929.4  (archive-exact)
[sandiego2] train 4455 px (99, 45)  test 2226 px (53, 42)  ||s||=27929.4
[sandiego2] seed 42


LRao s42:  11%|█▏        | 114/1000 [01:04<08:03,  1.83it/s, bad=19, train=-0.5754, val=-0.7073]

    [LRao] early stop at epoch 115 (no new val minimum for 20 epochs)


    [LRao] best val epoch 95 (val -0.9907, stopped 115, fit 3564/val 891)
    resumed repro/checkpoints/deep/THANTD_sandiego2_seed42.pt
    resumed repro/checkpoints/deep/HTDNet_sandiego2_seed42.pt
    resumed repro/checkpoints/deep/TSTTD_sandiego2_seed42.pt
    resumed repro/checkpoints/deep/OSVAE_sandiego2_seed42.pt


In [4]:
SP.summarize('results/spatial')

[pavia4] no metrics yet
[sandiego] no metrics yet
[sandiego2] no metrics yet


In [5]:
from repro.analysis.verify import verify_spatial
from repro.analysis.tables import make_tables
verify_spatial('results/spatial')
make_tables('results/spatial', dst='results/spatial/tables')

=== verification vs published numbers (θ=0.15) ===
Δ = fresh - published.  Training-free rows should be |Δ| < 0.002; trained rows within seed-level std.
[pavia4] no fresh metrics — skipped
[sandiego] no fresh metrics — skipped
[sandiego2] no fresh metrics — skipped


FileNotFoundError: [Errno 2] No such file or directory: 'results/spatial/pavia4/metrics.json'

In [ ]:
import zipfile, os
with zipfile.ZipFile('spatial_verification.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk('results/spatial'):
        for fn in files:
            z.write(os.path.join(root, fn))
print('zipped -> spatial_verification.zip')
try:
    from google.colab import files
    files.download('spatial_verification.zip')
except Exception as e:
    print('manual download:', e)